In [1]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn 
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, PowerTransformer

In [2]:
# Load datasets
df1 = pd.read_csv("observations2.tsv", sep='\t')
df2 = pd.read_csv("patients2.tsv", sep='\t')
df3 = pd.read_csv("stations2.tsv", sep='\t')

In [3]:
keys = ['longitude', 'latitude']
df = pd.merge(df1, df3, on=keys, how='inner')
df


,SPO2,HR,PI,RR,ETCO2,FIO2,PRV,BP,ST,MAI,...,Signal Quality Index,Respiratory effort,O₂ extraction ratio,SNR,oximetry,latitude,longitude,code,station,location
0,97.287434,72.727986,13.660737,15.755854,39.737731,49.035774,87.868263,99.655904,35.376585,7.955277,...,51.192423,38.744548,0.229549,25.836222,1.0,35.06544,1.049450,DZ,Frenda,Africa/Algiers
1,98.537124,73.435702,4.453073,15.925616,40.015592,76.884562,76.449362,99.778323,36.578250,10.290540,...,54.580531,44.654611,0.229253,22.657638,1.0,23.56540,119.586270,TW,Magong,Asia/Taipei
2,99.126826,74.484895,14.230313,16.793470,38.773612,65.426955,76.714834,111.320081,34.413950,7.645887,...,50.899240,36.904112,0.221017,30.960610,1.0,48.79325,2.292750,FR,Fontenay-aux-Roses,Europe/Paris
3,98.699062,72.789517,3.295218,16.537063,40.458715,54.228426,84.446027,100.136492,36.216960,12.364709,...,37.521845,65.873715,0.263626,34.126183,1.0,60.02427,30.284910,RU,Kolomyagi,Europe/Moscow
4,96.775490,81.687039,4.805640,15.542213,38.756204,67.977808,127.867499,104.221144,35.304788,13.230194,...,64.705098,52.729143,0.219447,21.006023,0.0,-5.85746,144.230580,PG,Mount Hagen,Pacific/Port_Moresby
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12067,97.053526,79.517234,9.261270,14.736852,39.620976,57.772883,116.205787,105.639263,35.526930,8.452890,...,45.838061,63.403682,0.210694,23.844717,1.0,1.65610,103.603200,MY,Kulai,Asia/Kuala_Lumpur
12068,97.539191,86.483856,8.669419,17.095280,39.822356,59.610411,131.395847,100.085537,34.911060,9.644136,...,56.692310,51.751882,0.203792,32.208986,0.0,49.73843,13.373637,CZ,Plzeň,Europe/Prague
12069,97.792523,81.619683,11.937089,15.706645,42.517437,48.626094,67.856358,108.645851,34.993891,8.981748,...,60.316394,31.344551,0.241141,35.746108,1.0,28.15112,-82.461480,US,Lutz,America/New_York
12070,97.221186,81.171025,6.265514,16.152893,39.785292,36.938033,133.152911,104.618399,34.828797,7.306094,...,59.732007,27.343396,0.258182,33.449893,0.0,50.80019,7.207690,DE,Siegburg,Europe/Berlin


In [4]:
TARGET = 'oximetry'

DROP_COLS = [
    'location'
]

X = df.drop(columns=[TARGET] + DROP_COLS)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training size: {X_train.shape}, Test size: {X_test.shape}")

Training size: (8450, 24), Test size: (3622, 24)


In [5]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Sklearn-compatible transformer for outlier clipping (winsorization),
    supporting per-column method selection (IQR or SD).
    """

    def __init__(self, methods_dict, sd_threshold=3, lower_pct=0.05, upper_pct=0.95, softness=0.99):
        """
        methods_dict: dict, e.g. {"HR": "SD", "SPO2": "IQR"}
        sd_threshold: cutoff for z-score when using SD
        lower_pct, upper_pct: quantile bounds used for soft winsorization
        softness: smoothing factor (0=no smoothing, 1=full)
        """
        self.methods_dict = methods_dict
        self.sd_threshold = sd_threshold
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct
        self.softness = softness

    def fit(self, X, y=None):
        """Compute winsorization bounds from TRAIN data only."""
        X = pd.DataFrame(X).copy()
        self.bounds_ = {}

        for col, method in self.methods_dict.items():
            data = X[col].dropna()
            if len(data) < 5:
                self.bounds_[col] = None
                continue

            # Hard bounds (IQR or SD)
            if method == "IQR":
                Q1, Q3 = data.quantile([0.25, 0.75])
                IQR = Q3 - Q1
                lower_hard = Q1 - 1.5 * IQR
                upper_hard = Q3 + 1.5 * IQR

            elif method == "SD":
                mean, std = data.mean(), data.std(ddof=1)
                lower_hard = mean - self.sd_threshold * std
                upper_hard = mean + self.sd_threshold * std

            else:
                raise ValueError(f"Unknown method {method}. Use 'IQR' or 'SD'.")

            # Soft clipping bounds
            lower_soft, upper_soft = data.quantile([self.lower_pct, self.upper_pct])

            self.bounds_[col] = {
                "lower_soft": lower_soft,
                "upper_soft": upper_soft,
                "lower_hard": lower_hard,
                "upper_hard": upper_hard
            }

        return self

    def transform(self, X):
        """Apply winsorization using bounds computed in fit()."""
        X = pd.DataFrame(X).copy()

        for col, bounds in self.bounds_.items():
            if bounds is None or col not in X.columns:
                continue

            ls, us = bounds["lower_soft"], bounds["upper_soft"]

            X[col] = np.where(
                X[col] < ls,
                ls + self.softness * (X[col] - ls),
                X[col]
            )
            X[col] = np.where(
                X[col] > us,
                us - self.softness * (us - X[col]),
                X[col]
            )

        return X


In [6]:
WINSOR_METHODS = {
    'SPO2': 'IQR',
    'HR': 'SD',
    'PI': 'IQR',
    'RR': 'SD',
    'FIO2': 'SD',
    'ETCO2': 'IQR',
    'MAI': 'IQR',
    'PRV': 'SD',
    'BP': 'SD',
    'ST': 'SD',
    'CO': 'IQR',
    'PVI': 'SD',
    'Hb level': 'SD',
    'SV': 'IQR',
    'Blood Flow Index': 'SD',
    'PPG waveform features': 'SD',
    'Signal Quality Index': 'SD',
    'Respiratory effort': 'SD',
    'O₂ extraction ratio': 'IQR',
    'SNR': 'IQR',
    'latitude': 'IQR',
    'longitude': 'IQR'
}
winsorizer = Winsorizer(methods_dict=WINSOR_METHODS)

In [7]:
STANDARD_NUM = [
    'HR','RR','FIO2','PRV','BP','PVI','Hb level',
    'Blood Flow Index','PPG waveform features',
    'Signal Quality Index','Respiratory effort'
]

ROBUST_NUM = [
    'SPO2','PI','ETCO2','ST','MAI','SV','latitude','longitude', 'O₂ extraction ratio', 'SNR'
]

POWER_NUM = ['CO']

LOW_CARD_CAT_COLS = [
    'code'        # 107 levels
]

HIGH_CARD_CAT_COLS = [
    'station'     # 552 levels
]

In [8]:
# NUMERIC PIPELINES

from sklearn.impute import SimpleImputer
from category_encoders.leave_one_out import LeaveOneOutEncoder

standard_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

robust_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

power_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("power", PowerTransformer(method="yeo-johnson")),
    ("scaler", StandardScaler())
])


# CATEGORICAL PIPELINES

low_card_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

high_card_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("leaveoneout", LeaveOneOutEncoder())
])



In [9]:
from sklearn.compose import ColumnTransformer


preprocessor = ColumnTransformer(
    transformers=[
        ("winsor", winsorizer, list(WINSOR_METHODS.keys())), 
        ("num_standard", standard_numeric_pipeline, STANDARD_NUM),
        ("num_robust", robust_numeric_pipeline, ROBUST_NUM),
        ("num_power", power_numeric_pipeline, POWER_NUM),
        ("lowcat", low_card_pipeline, LOW_CARD_CAT_COLS),
        ("highcat", high_card_pipeline, HIGH_CARD_CAT_COLS),
    ],
    remainder="drop"
)

In [10]:
#since wrapper model rfe is really time-consumming we replace him with randomForest less complex but still relatively eficient

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel


selector = SelectFromModel(
    RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    max_features=10
)


preprocessed_selector = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", selector)
])

X_train_processed = preprocessed_selector.fit_transform(X_train, y_train)
X_test_processed  = preprocessed_selector.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(8450, 10)
(3622, 10)


In [11]:
X_train_np = X_train_processed.values if hasattr(X_train_processed, "values") else X_train_processed
y_train_np = y_train.values.ravel() if hasattr(y_train, "values") else y_train

X_test_np = X_test_processed.values if hasattr(X_test_processed, "values") else X_test_processed
y_test_np = y_test.values.ravel() if hasattr(y_test, "values") else y_test

## ID3

ID3 is simple and clasical algorithm for building decision trees. It's 

In [12]:

def entropy(y):
    """
    Vypočíta entropiu cieľovej premenné y.
    y: 1D numpy array s triedami
    """
    values, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs + 1e-9))  # +1e-9 aby sa predišlo log(0)

def information_gain(y, x):
    """
    Vypočíta information gain pri rozdelení y podľa stĺpca x.
    y: cieľová premenná (numpy array)
    x: atribút podľa ktorého delíme (numpy array)
    """
    # entropia pred splitom
    H_y = entropy(y)
    
    # jedinečné hodnoty atribútu x
    values, counts = np.unique(x, return_counts=True)
    
    # entropia po splitu
    H_cond = 0
    for v, c in zip(values, counts):
        y_v = y[x == v]  # podmnožina y, kde x = v
        H_cond += (c / len(y)) * entropy(y_v)
    
    return H_y - H_cond

In [13]:
def id3(X, y, depth=0, max_depth=3):
    if np.all(y == y[0]):
        return {"type": "leaf", "class": y[0]}
    
    if depth >= max_depth or X.shape[1] == 0:
        values, counts = np.unique(y, return_counts=True)
        majority = values[np.argmax(counts)]
        return {"type": "leaf", "class": majority}

    best_index = None
    best_gain = -np.inf
    for i in range(X.shape[1]):
        gain = information_gain(y, X[:, i])
        if gain > best_gain:
            best_gain = gain
            best_index = i

    # if no attribute provides gain, create leaf with majority
    if best_index is None or best_gain <= 0:
        values, counts = np.unique(y, return_counts=True)
        majority = values[np.argmax(counts)]
        return {"type": "leaf", "class": majority}

    node = {"type": "node", "attribute": best_index, "children": {}}
    values = np.unique(X[:, best_index])
    for value in values:
        mask = X[:, best_index] == value
        X_subset = X[mask]
        y_subset = y[mask]

        if len(y_subset) == 0:
            values_all, counts_all = np.unique(y, return_counts=True)
            majority = values_all[np.argmax(counts_all)]
            child = {"type": "leaf", "class": majority}
        else:
            child = id3(X_subset, y_subset, depth+1, max_depth=max_depth)

        node["children"][value] = child

    return node

In [14]:
def predict(tree, x):
    if tree["type"] == "leaf":
        return tree["class"]
    
    attr = tree["attribute"]
    value = x[attr]
    
    if value in tree["children"]:
        return predict(tree["children"][value], x)
    
    # fallback if unseen value
    child_classes = [c["class"] for c in tree["children"].values() if c["type"]=="leaf"]
    if child_classes:
        return max(set(child_classes), key=child_classes.count)
    return None

In [15]:

max_depth = 2

tree = id3(X_train_np, y_train_np, depth=0, max_depth=max_depth)


In [16]:
y_pred_train = np.array([predict(tree, row) for row in X_train_np])

In [17]:
# predictions
y_pred = np.array([predict(tree, row) for row in X_test_np])

In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print("TRAIN accuracy:", accuracy_score(y_train_np, y_pred_train))
print("TRAIN precision (macro):", precision_score(y_train_np, y_pred_train, average='macro'))
print("TRAIN recall (macro):", recall_score(y_train_np, y_pred_train, average='macro'))
print("TRAIN confusion:\n", confusion_matrix(y_train_np, y_pred_train))

print()
print("TEST accuracy:", accuracy_score(y_test_np, y_pred))
print("TEST precision (macro):", precision_score(y_test_np, y_pred, average='macro'))
print("TEST recall (macro):", recall_score(y_test_np, y_pred, average='macro'))
print("TEST confusion:\n", confusion_matrix(y_test_np, y_pred))

TRAIN accuracy: 1.0
TRAIN precision (macro): 1.0
TRAIN recall (macro): 1.0
TRAIN confusion:
 [[3435    0]
 [   0 5015]]

TEST accuracy: 0.6032578685808946
TEST precision (macro): 0.7996374790853318
TEST recall (macro): 0.5122199592668024
TEST confusion:
 [[  36 1437]
 [   0 2149]]


In [19]:
def entropy(y):
    values, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs + 1e-9))


# ---------- Information Gain for categorical ----------
def information_gain_categorical(y, x):
    H_before = entropy(y)
    values, counts = np.unique(x, return_counts=True)

    H_after = 0
    for v, c in zip(values, counts):
        y_subset = y[x == v]
        H_after += (c / len(y)) * entropy(y_subset)

    return H_before - H_after


# ---------- Information Gain for numeric threshold ----------
def information_gain_numeric(y, x, threshold):
    left_mask = x <= threshold
    right_mask = x > threshold

    if left_mask.sum() == 0 or right_mask.sum() == 0:
        return -np.inf

    H_before = entropy(y)

    H_left = entropy(y[left_mask])
    H_right = entropy(y[right_mask])

    w_left = left_mask.sum() / len(y)
    w_right = right_mask.sum() / len(y)

    H_after = w_left * H_left + w_right * H_right

    return H_before - H_after

In [20]:
def candidate_thresholds(x):
    """Return midpoints between sorted unique values."""
    uniq = np.unique(x)
    if len(uniq) <= 1:
        return []
    return (uniq[:-1] + uniq[1:]) / 2.0

In [21]:
def id3(X, y, depth=0, max_depth=3):
    # Pure class → leaf
    if np.all(y == y[0]):
        return {"type": "leaf", "class": y[0]}

    # Max depth or no columns left → leaf
    if depth >= max_depth or X.shape[1] == 0:
        values, counts = np.unique(y, return_counts=True)
        majority = values[np.argmax(counts)]
        return {"type": "leaf", "class": majority}

    best_index = None
    best_gain = -np.inf
    best_threshold = None
    best_is_numeric = False

    # --- evaluate all attributes ---
    for i in range(X.shape[1]):
        column = X[:, i]

        # numeric → try thresholds
        if np.issubdtype(column.dtype, np.number):
            for t in candidate_thresholds(column):
                gain = information_gain_numeric(y, column, t)
                if gain > best_gain:
                    best_gain = gain
                    best_index = i
                    best_threshold = t
                    best_is_numeric = True

        # categorical → standard IG
        else:
            gain = information_gain_categorical(y, column)
            if gain > best_gain:
                best_gain = gain
                best_index = i
                best_threshold = None
                best_is_numeric = False

    # no useful split → leaf
    if best_index is None or best_gain <= 0:
        values, counts = np.unique(y, return_counts=True)
        majority = values[np.argmax(counts)]
        return {"type": "leaf", "class": majority}

    # ---------- numeric split ----------
    if best_is_numeric:
        xcol = X[:, best_index]
        left_mask = xcol <= best_threshold
        right_mask = xcol > best_threshold

        left_child = id3(X[left_mask], y[left_mask], depth+1, max_depth)
        right_child = id3(X[right_mask], y[right_mask], depth+1, max_depth)

        return {
            "type": "node",
            "attribute": best_index,
            "numeric": True,
            "threshold": best_threshold,
            "children": {
                "<=": left_child,
                ">": right_child
            }
        }

    # ---------- categorical split ----------
    node = {
        "type": "node",
        "attribute": best_index,
        "numeric": False,
        "threshold": None,
        "children": {}
    }

    values = np.unique(X[:, best_index])
    for v in values:
        mask = X[:, best_index] == v
        if mask.sum() == 0:
            # majority fallback
            values_all, counts_all = np.unique(y, return_counts=True)
            majority = values_all[np.argmax(counts_all)]
            child = {"type": "leaf", "class": majority}
        else:
            child = id3(X[mask], y[mask], depth+1, max_depth)
        node["children"][v] = child

    return node

In [22]:
def predict(tree, x):
    if tree["type"] == "leaf":
        return tree["class"]

    attr = tree["attribute"]

    # numeric split
    if tree["numeric"]:
        if x[attr] <= tree["threshold"]:
            return predict(tree["children"]["<="], x)
        else:
            return predict(tree["children"][">"], x)

    # categorical
    value = x[attr]
    if value in tree["children"]:
        return predict(tree["children"][value], x)
    else:
        # unseen category → cannot decide → return majority fallback?
        return None

In [23]:
tree = id3(X_train_np, y_train_np, max_depth=2)

In [24]:
y_pred = np.array([predict(tree, row) for row in X_test_np])

In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print()
print("TEST accuracy:", accuracy_score(y_test_np, y_pred))
print("TEST precision (macro):", precision_score(y_test_np, y_pred, average='macro'))
print("TEST recall (macro):", recall_score(y_test_np, y_pred, average='macro'))
print("TEST confusion:\n", confusion_matrix(y_test_np, y_pred))


TEST accuracy: 0.7954168967421315
TEST precision (macro): 0.8021280111633373
TEST recall (macro): 0.8118980172656443
TEST confusion:
 [[1326  147]
 [ 594 1555]]
